# 02 Grid And Isochrones

本 notebook 生成 500 米网格，构建四种出行方式的 15 分钟近似等时圈阈值，并把基础层与 Track A POI 的空间可达计数缓存到网格层。这里采用批量半径近似，而不是为全市每个网格逐一下载真实路网等时圈，以控制运行成本并确保课程提交可复现。


In [ ]:
from pathlib import Path
import math

import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

SUBMIT_DIR = ROOT / "submit"
CACHE_DIR = SUBMIT_DIR / "cache"
OUTPUT_DIR = SUBMIT_DIR / "output"

districts = gpd.read_file(CACHE_DIR / "boundaries.gpkg", layer="districts").to_crs("EPSG:4326")
towns = gpd.read_file(CACHE_DIR / "boundaries.gpkg", layer="towns").to_crs("EPSG:4326")
poi = pd.read_parquet(CACHE_DIR / "poi_base_tracka_clean.parquet")


In [ ]:
PROJECTED_CRS = "EPSG:32651"
GRID_SIZE_M = 500
MODE_RADII = {
    "walk": 1200,
    "bike": 3500,
    "transit": 4000,
    "drive": 6000,
}

city = districts.dissolve().to_crs(PROJECTED_CRS)
minx, miny, maxx, maxy = city.total_bounds

xs = np.arange(minx, maxx + GRID_SIZE_M, GRID_SIZE_M)
ys = np.arange(miny, maxy + GRID_SIZE_M, GRID_SIZE_M)
cells = []
gid = 0
for x in xs[:-1]:
    for y in ys[:-1]:
        poly = gpd.GeoSeries.from_wkt([f"POLYGON (({x} {y}, {x + GRID_SIZE_M} {y}, {x + GRID_SIZE_M} {y + GRID_SIZE_M}, {x} {y + GRID_SIZE_M}, {x} {y}))"], crs=PROJECTED_CRS).iloc[0]
        cells.append({"grid_id": gid, "geometry": poly})
        gid += 1

grid = gpd.GeoDataFrame(cells, geometry="geometry", crs=PROJECTED_CRS)
grid = gpd.overlay(grid, city[["geometry"]], how="intersection", keep_geom_type=True)
grid["centroid"] = grid.geometry.centroid
grid_wgs84 = grid.to_crs("EPSG:4326")
grid_wgs84["center_lon"] = grid_wgs84["centroid"].to_crs("EPSG:4326").x
grid_wgs84["center_lat"] = grid_wgs84["centroid"].to_crs("EPSG:4326").y
len(grid_wgs84)


In [ ]:
centers = gpd.GeoDataFrame(
    grid_wgs84[["grid_id", "center_lon", "center_lat"]].copy(),
    geometry=gpd.points_from_xy(grid_wgs84["center_lon"], grid_wgs84["center_lat"]),
    crs="EPSG:4326",
)

centers = gpd.sjoin(centers, districts[["district_name", "district_code", "geometry"]], predicate="within", how="left").drop(columns=["index_right"])
centers = gpd.sjoin(centers, towns[["town_name", "town_code", "geometry"]], predicate="within", how="left").drop(columns=["index_right"])
centers.head()


In [ ]:
grid_cache = CACHE_DIR / "grid_500m.gpkg"
grid_wgs84.drop(columns=["centroid"]).to_file(grid_cache, layer="grid", driver="GPKG")
centers[["grid_id", "center_lon", "center_lat", "district_name", "district_code", "town_name", "town_code", "geometry"]].to_file(
    grid_cache, layer="grid_centers", driver="GPKG"
)
grid_cache


In [ ]:
poi_gdf = gpd.GeoDataFrame(
    poi.copy(),
    geometry=gpd.points_from_xy(poi["wgs84Lng"], poi["wgs84Lat"]),
    crs="EPSG:4326",
).to_crs(PROJECTED_CRS)

center_proj = centers.to_crs(PROJECTED_CRS)
center_xy = np.column_stack([center_proj.geometry.x.to_numpy(), center_proj.geometry.y.to_numpy()])


In [ ]:
access = centers.drop(columns="geometry").copy()
indicator_keys = sorted(poi["indicator_key"].unique())

for indicator in indicator_keys:
    subset = poi_gdf[poi_gdf["indicator_key"] == indicator]
    poi_xy = np.column_stack([subset.geometry.x.to_numpy(), subset.geometry.y.to_numpy()])
    tree = cKDTree(poi_xy)
    for mode, radius in MODE_RADII.items():
        counts = tree.query_ball_point(center_xy, r=radius, return_length=True)
        access[f"{mode}__{indicator}"] = np.asarray(counts, dtype=int)

access.head()


In [ ]:
access_cache = CACHE_DIR / "grid_access_counts.parquet"
access.to_parquet(access_cache, index=False)

cache_note = {
    "grid_cache": str(grid_cache),
    "access_cache": str(access_cache),
    "mode_radii_meters": MODE_RADII,
    "cache_strategy": [
        "边界、绿地、POI 先清洗后缓存，避免重复 IO",
        "网格中心点与行政归属单独缓存，便于后续重算不同评分方案",
        "可达计数按 mode__indicator_key 命名，直接供评分 notebook 调用"
    ]
}
cache_note


## 网格、等时圈与陆地掩膜

本 notebook 对应交付要求：500 米网格、四种出行方式等时圈、空间连接和缓存策略。500 米网格作为中间分析单元，步行、自行车、公交、驾车四种 15 分钟阈值用于近似全市尺度可达范围。由于上海行政区边界包含长江口、杭州湾和近海范围，如果直接对行政区面生成 H3，会把大量海域格网纳入评分，因此最终 H3 输出前加入陆地过滤策略。

`sports desert.py` 和 `applicationV1` 采用同一套陆地过滤口径：

1. 先按行政区 union 生成 H3 resolution 8 候选六边形；
2. 读取 `data/10-AI解译数据/*建筑.shp` 作为可居住陆域代理；
3. 用 H3 polygon 与建筑多边形做 `intersects` 空间连接；
4. 只保留命中的 H3，海域、江面和无建筑水面不进入最终网页数据；
5. 网页端优先复用 `outputs/sports_desert/sports_desert_h3.parquet` 中已经过滤过的 H3 列表，缺失时再回退到建筑掩膜即时计算。


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

sports_h3_path = ROOT / "outputs" / "sports_desert" / "sports_desert_h3.parquet"
app_json_path = ROOT / "submit" / "applicationV1" / "public" / "data" / "app_v1_h3_data_compact.json"
ai_buildings = sorted((ROOT / "data" / "10-AI解译数据").glob("**/*AI解译建筑.shp")) + sorted((ROOT / "data" / "10-AI解译数据").glob("**/*Ai解译建筑.shp"))

validation = {
    "ai_building_mask_file_count": len(ai_buildings),
    "sports_desert_land_h3_path": str(sports_h3_path),
    "sports_desert_land_h3_exists": sports_h3_path.exists(),
    "application_v1_json_exists": app_json_path.exists(),
}

if sports_h3_path.exists():
    try:
        sports_h3 = pd.read_parquet(sports_h3_path, columns=["h3"])
        validation["land_h3_count_from_sports_desert"] = int(sports_h3["h3"].nunique())
    except Exception as exc:
        validation["sports_desert_read_error"] = str(exc)

if app_json_path.exists():
    payload = json.loads(app_json_path.read_text(encoding="utf-8"))
    validation["application_v1_h3_count"] = int(len(payload.get("rows", [])))
    validation["application_v1_h3_resolution"] = 8

validation


In [ ]:
# 该片段对应 sports desert.py / scripts/build_application_v1_data.py 的核心逻辑，供复现实验时核对。
land_mask_algorithm = """
候选 H3：h3.geo_to_cells(city_polygon, resolution=8)
H3 polygon：h3.cell_to_boundary(cell) -> shapely Polygon
陆地代理：data/10-AI解译数据/*建筑.shp 合并为 building land_mask
空间过滤：geopandas.sjoin(hexes[['h3', 'geometry']], land_mask, predicate='intersects', how='inner')
最终保留：hexes[hexes['h3'].isin(on_land['h3'].unique())]
"""
print(land_mask_algorithm)
